# 02 — PRNU Extraction & NCC Baseline

Extract PRNU fingerprints and run NCC-based device identification.
Covers Experiments A1 (clean-domain) and B2 (wavelet vs Wiener).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import time

from src.prnu_pipeline import PRNUPipeline
from src.algorithms.ncc_baseline import ncc_score, ncc_score_batch, compute_threshold
from src.utils.helpers import wavelet_denoise, wiener_denoise, jpeg_compress
from src.utils.evaluation_metrics import compute_far_frr, compute_eer

%matplotlib inline

In [ ]:
# Create synthetic multi-device scenario
N_DEVICES = 5
N_TRAIN = 20
N_TEST = 5
IMG_SIZE = 256

pipeline = PRNUPipeline(denoiser='wavelet')
rng = np.random.RandomState(42)

# Generate unique PRNU for each device
device_names = [f'Camera_{i}' for i in range(N_DEVICES)]
device_prnus = {}
for i, name in enumerate(device_names):
    device_prnus[name] = rng.normal(0, 0.01, (IMG_SIZE, IMG_SIZE, 3))

print(f'Created {N_DEVICES} synthetic cameras')

In [ ]:
# Estimate fingerprints from training images
fingerprints = {}
for name, prnu in device_prnus.items():
    train_images = []
    for _ in range(N_TRAIN):
        scene = rng.randint(50, 200, (IMG_SIZE, IMG_SIZE, 3)).astype(np.float64) / 255.0
        img = np.clip(scene * (1 + prnu) + rng.normal(0, 0.005, scene.shape), 0, 1)
        train_images.append((img * 255).astype(np.uint8))
    
    fingerprints[name] = pipeline.estimate_fingerprint(train_images, show_progress=False)
    print(f'  {name}: fingerprint estimated from {N_TRAIN} images')

print(f'\nAll {N_DEVICES} fingerprints estimated')

In [ ]:
# Test identification with NCC
correct = 0
total = 0

for true_device, prnu in device_prnus.items():
    for _ in range(N_TEST):
        scene = rng.randint(50, 200, (IMG_SIZE, IMG_SIZE, 3)).astype(np.float64) / 255.0
        img = np.clip(scene * (1 + prnu) + rng.normal(0, 0.005, scene.shape), 0, 1)
        query = (img * 255).astype(np.uint8)
        
        scores = pipeline.identify(query, fingerprints)
        predicted = list(scores.keys())[0]
        
        if predicted == true_device:
            correct += 1
        total += 1

accuracy = correct / total * 100
print(f'NCC Baseline Accuracy: {accuracy:.1f}% ({correct}/{total})')

In [ ]:
# Compare wavelet vs Wiener denoiser (Experiment B2)
for denoiser_name in ['wavelet', 'wiener']:
    p = PRNUPipeline(denoiser=denoiser_name)
    correct = 0
    total = 0
    
    # Re-estimate fingerprints with this denoiser
    fps = {}
    for name, prnu in device_prnus.items():
        imgs = []
        for _ in range(N_TRAIN):
            scene = rng.randint(50, 200, (IMG_SIZE, IMG_SIZE, 3)).astype(np.float64) / 255.0
            img = np.clip(scene * (1 + prnu) + rng.normal(0, 0.005, scene.shape), 0, 1)
            imgs.append((img * 255).astype(np.uint8))
        fps[name] = p.estimate_fingerprint(imgs, show_progress=False)
    
    for true_device, prnu in device_prnus.items():
        for _ in range(N_TEST):
            scene = rng.randint(50, 200, (IMG_SIZE, IMG_SIZE, 3)).astype(np.float64) / 255.0
            img = np.clip(scene * (1 + prnu) + rng.normal(0, 0.005, scene.shape), 0, 1)
            scores = p.identify((img * 255).astype(np.uint8), fps)
            if list(scores.keys())[0] == true_device:
                correct += 1
            total += 1
    
    print(f'{denoiser_name:>8}: {correct/total*100:.1f}% accuracy')